In [1]:
import pandas as pd
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

In [3]:
df = sns.load_dataset('titanic')
print("--- PRIMEROS DATOS DEL TITANIC ---")
print(df[['survived','pclass','sex','age','fare']].head())

--- PRIMEROS DATOS DEL TITANIC ---
   survived  pclass     sex   age     fare
0         0       3    male  22.0   7.2500
1         1       1  female  38.0  71.2833
2         1       3  female  26.0   7.9250
3         1       1  female  35.0  53.1000
4         0       3    male  35.0   8.0500


In [4]:
X = df[['pclass','sex','age','fare']]
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)
print(f"\nEntrenaremos con {len(X_train)} pasajeros y evaluamos con {len(X_test)}.")


Entrenaremos con 712 pasajeros y evaluamos con 179.


In [5]:
# Definimos que hacer con los numeros (edad y tarifa)
transformer_numerico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])
# Definimos que hacer con las categorias (sexo)
transformer_categorico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
# Unimos ambos procesos en un preprocesador
preprocesador = ColumnTransformer(transformers=[
    ('num', transformer_numerico, ['age','fare']),
    ('cat', transformer_categorico, ['sex', 'pclass'])
])
# Creamos pipeline final
pipeline_titanic = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('clasificador', RandomForestClassifier(n_estimators=100, random_state=42))
])

In [7]:
print("\n --- Entrenando el modelo del Titanic ---")
pipeline_titanic.fit(X_train, y_train)
predicciones = pipeline_titanic.predict(X_test)
print("\n --- Informe de clasificacion ---")
print(classification_report(y_test, predicciones))


 --- Entrenando el modelo del Titanic ---

 --- Informe de clasificacion ---
              precision    recall  f1-score   support

           0       0.81      0.84      0.83       105
           1       0.76      0.73      0.74        74

    accuracy                           0.79       179
   macro avg       0.79      0.78      0.79       179
weighted avg       0.79      0.79      0.79       179



In [8]:
joblib.dump(pipeline_titanic, 'modelo_titanic.pkl')
print("\n Modelo guardado con exito")


 Modelo guardado con exito


In [9]:
# Cargamos el modelo
modelo_cargado = joblib.load('modelo_titanic.pkl')
# Creamos pasajeros ficticios
pasajeros_nuevos = pd.DataFrame([
    {'pclass':1, 'sex':'female','age':22,'fare':150},
    {'pclass':3, 'sex':'male','age':25,'fare':7.5},
    {'pclass':2, 'sex':'male','age':60,'fare':20}
])
resultados = modelo_cargado.predict(pasajeros_nuevos)
probabilidades = modelo_cargado.predict_proba(pasajeros_nuevos)
print("\n --- Predicciones del destino ---")
for i, p in enumerate(resultados):
    estado = "Sobrevive" if p == 1 else "No sobrevive"
    seguridad = probabilidades[i][p]*100
    print(f"Pasajero {i+1}: {estado} (Seguridad: {seguridad:.2f}%)")


 --- Predicciones del destino ---
Pasajero 1: Sobrevive (Seguridad: 83.00%)
Pasajero 2: No sobrevive (Seguridad: 98.00%)
Pasajero 3: No sobrevive (Seguridad: 96.00%)
